## 4. Missing Value Treatment

In [ ]:
missing_stats = df_cleaned.isnull().mean() * 100
missing_stats = missing_stats[missing_stats > 0].sort_values(ascending=False)
print("Features with Missing Data (%):")
print(missing_stats.head(25))

### Missing Value Treatment Decisions

| Feature Group | Missing % | Interpretation | Treatment |
|---|---|---|---|
| Joint application (`sec_app_*`) | ~98% | Non-joint loans: no co-borrower | Create `is_joint_application` flag; drop `sec_app_*` |
| Derogatory history (`mths_since_*`) | 66–83% | Missing = event never occurred | Fill 999 sentinel; create `ever_*` binary flag |
| Installment utilization (`il_util`) | 60–65% | Missing = no installment accounts | Create `has_installment_account`; fill 0 |
| Loan description (`desc`) | 91% | Optional free text | Create `has_desc` presence flag |

**Key insight:** In credit modeling, *missing = never* is often the correct interpretation for derogatory fields. A borrower with `mths_since_last_major_derog = NaN` has no derogatory history — that is good credit behavior, not missing data.

In [ ]:
# GROUP 1: Joint application
df_cleaned['is_joint_application'] = df_cleaned['annual_inc_joint'].notna().astype(int)
joint_cols = [c for c in df_cleaned.columns if c.startswith('sec_app_')]
joint_cols += ['verification_status_joint', 'dti_joint', 'annual_inc_joint', 'revol_bal_joint']
df_cleaned.drop(columns=[col for col in joint_cols if col in df_cleaned.columns], inplace=True)

# GROUP 2: "Never happened" derogatory fields
never_event_cols = [
    'mths_since_last_record', 'mths_since_recent_bc_dlq',
    'mths_since_last_major_derog', 'mths_since_recent_revol_delinq'
]
for col in never_event_cols:
    if col in df_cleaned.columns:
        df_cleaned[f'ever_{col.replace("mths_since_", "")}'] = df_cleaned[col].notna().astype(int)
        df_cleaned[col] = df_cleaned[col].fillna(999)

# GROUP 3: Installment/utilization
if 'il_util' in df_cleaned.columns:
    df_cleaned['has_installment_account'] = df_cleaned['il_util'].notna().astype(int)
    df_cleaned['il_util'] = df_cleaned['il_util'].fillna(0)
if 'all_util' in df_cleaned.columns:
    df_cleaned['all_util'] = df_cleaned['all_util'].fillna(0)
if 'open_acc_6m' in df_cleaned.columns:
    df_cleaned['open_acc_6m'] = df_cleaned['open_acc_6m'].fillna(0)
if 'mths_since_rcnt_il' in df_cleaned.columns:
    df_cleaned['ever_had_installment'] = df_cleaned['mths_since_rcnt_il'].notna().astype(int)
    df_cleaned['mths_since_rcnt_il'] = df_cleaned['mths_since_rcnt_il'].fillna(999)

# GROUP 4: desc
if 'desc' in df_cleaned.columns:
    df_cleaned['has_desc'] = df_cleaned['desc'].notna().astype(int)

print("Missing value transformations complete.")
print(f"Current workspace shape: {df_cleaned.shape}")